# AgroPulse · Olive leaf disease model (R&D)

**Goal:** train our own image model that tells a healthy olive leaf from **peacock spot** (*Spilocaea oleaginea*, round spots on the leaf) and from **Aculus olearius** mite damage, and measure honestly how well it works on photos it has never seen.

> Run this notebook in **Google Colab with a GPU** (Runtime → Change runtime type → T4 GPU). It takes about 40–70 minutes (v1 and the v2 experiment).
> Everything it produces (figures, models, model card, tables) is also copied to **Google Drive: `MyDrive/AgroPulse/outputs/`**, so nothing is lost if Colab disconnects.

| Step | What happens |
|---|---|
| 1. Data | Two public datasets, class counts, sample images, train/validation/test split, duplicate check |
| 2. Model | EfficientNetB0 transfer learning, augmentation, class weights, two training phases |
| 3. Evaluation | Official Denizli test set, external Edincik test, Greek field photos (if available), Grad-CAM, examples, confidence |
| 3.8 Experiment v2 | Letterbox preprocessing + stronger colour augmentation, retrained and compared with v1 |
| 4. Baseline | Optional: a vision LLM on 60 test images (needs an API key) |
| 5. Export | Keras + TFLite of the better version, model card, figures |

## 0. Setup

Mounts Google Drive and unzips the two datasets from `MyDrive/AgroPulse/data/olive_leaves.zip` and `MyDrive/AgroPulse/data/edincik.zip` to the Colab disk.
Optional: Greek phone photos in `MyDrive/AgroPulse/data/field_photos/<class>/`.

In [ ]:
import json
import os
import random
import re
import shutil
import sys
import time
import warnings
import zipfile
from pathlib import Path

warnings.filterwarnings('ignore')
IN_COLAB = 'google.colab' in sys.modules
SEED = 42
random.seed(SEED)

DRIVE_ROOT = Path('/content/drive/MyDrive/AgroPulse')
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA = Path('/content/data')
    DATA.mkdir(exist_ok=True)
    for name in ['olive_leaves', 'edincik']:
        zip_path = DRIVE_ROOT / 'data' / f'{name}.zip'
        dest = DATA / name
        if dest.exists() and any(dest.rglob('*')):
            print(f'{name}: already unzipped')
        elif zip_path.exists():
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(dest)
            print(f'{name}: unzipped {zip_path.name}')
        else:
            print(f'{name}: MISSING {zip_path}')
    OUT_DRIVE = DRIVE_ROOT / 'outputs'
    OUT_DRIVE.mkdir(parents=True, exist_ok=True)
    REPO = Path('/content/research')
    FIELD_PHOTOS = DRIVE_ROOT / 'data' / 'field_photos'
else:
    # Running locally from the project folder.
    REPO = Path.cwd() if Path.cwd().name == 'research' else Path('research')
    DATA = REPO / 'data'
    OUT_DRIVE = None
    FIELD_PHOTOS = DATA / 'field_photos'

FIG_DIR = REPO / 'figures' / 'leaf'   # the weather figures of notebook 01 stay in research/figures/
MODEL_DIR = REPO / 'models'
# Trained models are saved here (on Drive in Colab). A later run loads them instead of training again.
# Set RETRAIN = True to train again even when checkpoints exist.
CHECKPOINT_DIR = (DRIVE_ROOT / 'checkpoints') if IN_COLAB else (REPO / 'checkpoints')
RETRAIN = False
for d in (FIG_DIR, MODEL_DIR, CHECKPOINT_DIR):
    d.mkdir(parents=True, exist_ok=True)
print('Data folder:', DATA.resolve())
print('Outputs:', REPO.resolve(), '| Drive outputs:', OUT_DRIVE)

In [ ]:
%pip install -q imagehash

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps

# Slide-friendly figures: 16:9, 200 dpi, white background.
FIGSIZE = (16, 9)
mpl.rcParams.update({
    'figure.figsize': FIGSIZE, 'figure.dpi': 80, 'savefig.dpi': 200, 'savefig.facecolor': 'white',
    'font.size': 15, 'axes.titlesize': 17, 'axes.titleweight': 'bold', 'axes.labelsize': 15,
    'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False,
})
SUBTITLE = 'AgroPulse olive leaf model · EfficientNetB0 transfer learning · Denizli + Edincik datasets'


def mirror(path, sub):
    """Copy a file to the Drive outputs folder (when running in Colab)."""
    if OUT_DRIVE is not None:
        target = OUT_DRIVE / sub
        target.mkdir(parents=True, exist_ok=True)
        shutil.copy(path, target / Path(path).name)


def headline(fig, title, subtitle=SUBTITLE, top=0.86):
    fig.subplots_adjust(top=top)
    fig.text(0.5, 0.965, title, ha='center', va='top', fontsize=26, fontweight='bold')
    fig.text(0.5, 0.905, subtitle, ha='center', va='top', fontsize=13, color='#6B6B6B')


def save_fig(fig, name):
    path = FIG_DIR / name
    fig.savefig(path)
    mirror(path, 'figures/leaf')
    plt.show()
    return path

## 1. Data

### 1.1 Sources, licence and citation

> **Still to add before showing this to the jury:** the two Kaggle page URLs (`[paste URL]`). Keep the licence text exactly as shown on the dataset pages.

| | Denizli dataset | Edincik dataset |
|---|---|---|
| Place | Denizli, Türkiye | Edincik, Bandırma, Balıkesir, Türkiye |
| Classes | Healthy, Olive Peacock Spot, Aculus Olearius | Healthy, Peacock spot |
| Images | 3,400 (official train/test split) | 954 (white background, light box) |
| Source | Kaggle, "Olive Leaf Image Dataset" | Kaggle, "Olive Leaf Disease" |
| Source URL | `[paste URL]` | `[paste URL]` |
| Licence | CC0 1.0 | unknown |
| Authors / citation | `[not provided]` | Diker, A., Elen, A., Közkurt, C. et al. Eur Food Res Technol (2023). https://doi.org/10.1007/s00217-023-04386-8 |
| Date downloaded | 29 Oct 2026 | 29 Oct 2026 |
| Use in this project | training, validation, test | external test only, **not used for training** |

Because the Edincik licence is unknown, its images are only used for testing here and are not redistributed (not in git, not in the Hugging Face Space).

### 1.2 Find the folders

The Kaggle zips can have an extra top-level folder or different class names. The cell below looks at the real structure and finds the train/test folders and the class folders by itself.

In [ ]:
IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
SPLIT_NAMES = {'train': 'train', 'training': 'train', 'test': 'test', 'testing': 'test',
               'val': 'val', 'valid': 'val', 'validation': 'val'}

# Our 3 classes. Order = label number.
CLASSES = ['Healthy', 'Olive Peacock Spot', 'Aculus Olearius']
CLASS_ID = {c: i for i, c in enumerate(CLASSES)}


def normalize_class(folder_name, two_class_dataset=False):
    """Folder name -> one of our class names (or None when we do not know it)."""
    n = re.sub('[^a-z]', '', folder_name.lower())
    if 'healthy' in n or 'saglikli' in n or n == 'normal':
        return 'Healthy'
    if 'peacock' in n or 'spot' in n or 'cycloconium' in n or 'gozlek' in n:
        return 'Olive Peacock Spot'
    if 'aculus' in n or 'olearius' in n:
        return 'Aculus Olearius'
    if two_class_dataset and any(k in n for k in ('diseas', 'sick', 'hasta', 'infect')):
        return 'Olive Peacock Spot'
    return None


def scan_images(root):
    """All images below `root` with the split (train/test/val, if any) and the class folder name."""
    rows = []
    root = Path(root)
    for p in root.rglob('*'):
        if not p.is_file() or p.suffix.lower() not in IMG_EXT or p.name.startswith('._') or '__MACOSX' in p.parts:
            continue
        parts = [x.lower() for x in p.relative_to(root).parts[:-1]]
        split = next((SPLIT_NAMES[x] for x in parts if x in SPLIT_NAMES), None)
        rows.append((str(p), split, p.parent.name))
    return pd.DataFrame(rows, columns=['path', 'split', 'class_raw'])


def show_structure(df, title):
    print(title)
    print(df.groupby(['split', 'class_raw'], dropna=False).size().rename('images').to_string() if len(df) else '  (no images found)')


den_raw = scan_images(DATA / 'olive_leaves')
edi_raw = scan_images(DATA / 'edincik')
show_structure(den_raw, 'Denizli folders found:')
print()
show_structure(edi_raw, 'Edincik folders found:')

In [ ]:
# Denizli: official train/test split, 3 classes.
den = den_raw.assign(label=den_raw.class_raw.map(normalize_class))
unknown = den[den.label.isna()].class_raw.unique()
if len(unknown):
    raise ValueError(f'Unknown Denizli class folders: {list(unknown)}. Add them to normalize_class().')
if den.split.isna().any():
    raise ValueError('Some Denizli images are not inside a train/ or test/ folder. Check the unzipped structure above.')
den['y'] = den.label.map(CLASS_ID)

# Edincik: 2 classes, no official split, used only as an external test.
EDINCIK_CLASS_OVERRIDE = {}   # for example {'my_folder_name': 'Healthy'} if the folder names are unusual
if len(edi_raw):
    edi = edi_raw.assign(label=edi_raw.class_raw.map(lambda n: EDINCIK_CLASS_OVERRIDE.get(n) or normalize_class(n, two_class_dataset=True)))
    unknown = edi[edi.label.isna()].class_raw.unique()
    if len(unknown):
        raise ValueError(f'Unknown Edincik class folders: {list(unknown)}. Add them to EDINCIK_CLASS_OVERRIDE.')
    edi = edi[edi.label.isin(['Healthy', 'Olive Peacock Spot'])].copy()
    edi['y'] = edi.label.map(CLASS_ID)
else:
    edi = pd.DataFrame(columns=['path', 'split', 'class_raw', 'label', 'y'])
    print('Edincik images not found: the external test 1 will be skipped.')

train_all = den[den.split == 'train'].reset_index(drop=True)
test = den[den.split == 'test'].reset_index(drop=True)
print(f'Denizli: {len(train_all)} train, {len(test)} test ({len(train_all) + len(test)} images; expected 3,400)')
print(f'Edincik: {len(edi)} images')

### 1.3 Class counts and sample images

In [ ]:
counts = pd.DataFrame({
    'Denizli train': train_all.label.value_counts(),
    'Denizli test': test.label.value_counts(),
    'Edincik': edi.label.value_counts(),
}).reindex(CLASSES).fillna(0).astype(int)
counts.loc['Total'] = counts.sum()
display(counts)

fig, ax = plt.subplots(figsize=FIGSIZE)
plot = counts.drop(index='Total')
x = np.arange(len(plot))
for k, (col, color) in enumerate(zip(plot.columns, ['#3F6B2F', '#2E6FB0', '#E0A526'])):
    bars = ax.bar(x + (k - 1) * 0.27, plot[col], 0.27, color=color, label=col)
    ax.bar_label(bars, fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(plot.index, fontsize=16)
ax.set_ylabel('Images')
ax.legend()
headline(fig, 'Images per class', top=0.84)
save_fig(fig, 'ds_class_counts.png');

In [ ]:
def show_samples(df, per_class=9, title='Sample images', name='ds_samples.png', classes=CLASSES):
    classes = [c for c in classes if (df.label == c).any()]
    fig, axes = plt.subplots(len(classes), per_class, figsize=FIGSIZE)
    axes = np.atleast_2d(axes)
    rng = np.random.default_rng(SEED)
    for r, c in enumerate(classes):
        sub = df[df.label == c]
        pick = sub.sample(min(per_class, len(sub)), random_state=SEED)
        for k in range(per_class):
            ax = axes[r, k]
            ax.axis('off')
            if k < len(pick):
                ax.imshow(ImageOps.exif_transpose(Image.open(pick.iloc[k].path).convert('RGB')))
        axes[r, 0].set_title(c, loc='left', fontsize=16, pad=6)
    headline(fig, title, top=0.86)
    fig.subplots_adjust(hspace=0.25, wspace=0.03, left=0.02, right=0.98, bottom=0.02)
    save_fig(fig, name)


show_samples(train_all, title='Denizli training images: 9 per class', name='ds_samples_denizli.png')
if len(edi):
    show_samples(edi, title='Edincik images (light box, white background): 9 per class', name='ds_samples_edincik_photos.png')

### 1.4 Train / validation / test split

* The **official Denizli test set stays untouched**: it is used once, at the end.
* **15 % of the training images** become the validation set (stratified by class, fixed seed). The model never trains on them. They are used to stop training at the right time.

In [ ]:
from sklearn.model_selection import train_test_split

tr_idx, val_idx = train_test_split(np.arange(len(train_all)), test_size=0.15, stratify=train_all.y, random_state=SEED)
train = train_all.iloc[tr_idx].reset_index(drop=True)
val = train_all.iloc[val_idx].reset_index(drop=True)
split_table = pd.DataFrame({'Train': train.label.value_counts(), 'Validation': val.label.value_counts(), 'Test (official)': test.label.value_counts()}).reindex(CLASSES)
split_table.loc['Total'] = split_table.sum()
display(split_table)

### 1.5 Duplicate check between train and test

Photos of the same leaf (or the same photo saved twice) in both train and test would make the test score look better than it is. We check:
* **exact duplicates**: identical file content (MD5);
* **near duplicates**: a *perceptual hash* (pHash) of 64 bits; two images whose hashes differ in at most 5 bits look the same to a person (resized, re-compressed or slightly cropped copies).

In [ ]:
import hashlib
from concurrent.futures import ThreadPoolExecutor

import imagehash

NEAR_BITS = 5


def file_md5(path):
    return hashlib.md5(Path(path).read_bytes()).hexdigest()


def phash_int(path):
    img = ImageOps.exif_transpose(Image.open(path)).convert('L')
    return int(str(imagehash.phash(img)), 16)


def compute(fn, paths):
    with ThreadPoolExecutor(max_workers=8) as ex:
        return list(ex.map(fn, paths))


t0 = time.time()
train_md5, test_md5 = compute(file_md5, train_all.path), compute(file_md5, test.path)
train_ph = np.array(compute(phash_int, train_all.path), dtype=np.uint64)
test_ph = np.array(compute(phash_int, test.path), dtype=np.uint64)

# Hamming distance between every test hash and every train hash.
POPCOUNT = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)
xor = np.bitwise_xor(test_ph[:, None], train_ph[None, :])
dist = POPCOUNT[xor.view(np.uint8).reshape(len(test_ph), len(train_ph), 8)].sum(-1)
nearest = dist.min(axis=1)

exact_dup = np.isin(test_md5, train_md5)
same_hash = nearest == 0
near_dup = nearest <= NEAR_BITS
test['near_dup'] = near_dup

dup_table = pd.DataFrame({
    'Test images': [len(test)] * 3,
    'Found in train': [int(exact_dup.sum()), int(same_hash.sum()), int(near_dup.sum())],
}, index=['Exact duplicate (same file content)', 'Same perceptual hash (0 bits apart)', f'Near duplicate (≤ {NEAR_BITS} bits apart)'])
dup_table['Share'] = (dup_table['Found in train'] / dup_table['Test images'] * 100).round(1).astype(str) + ' %'
display(dup_table)
print(f'Done in {time.time() - t0:.0f} s. Test images with a near duplicate in train: {int(near_dup.sum())}.')
print('Near duplicates by class:', test[test.near_dup].label.value_counts().to_dict())

In [ ]:
# Show up to 4 nearest pairs.
if near_dup.any():
    order = np.argsort(nearest)[: min(4, int(near_dup.sum()))]
    fig, axes = plt.subplots(2, len(order), figsize=FIGSIZE)
    axes = np.atleast_2d(axes).reshape(2, -1)
    for k, ti in enumerate(order):
        tj = int(dist[ti].argmin())
        axes[0, k].imshow(ImageOps.exif_transpose(Image.open(test.path[ti]).convert('RGB')))
        axes[0, k].set_title(f'Test · {test.label[ti]}', fontsize=13)
        axes[1, k].imshow(ImageOps.exif_transpose(Image.open(train_all.path[tj]).convert('RGB')))
        axes[1, k].set_title(f'Train · {int(dist[ti, tj])} bits apart', fontsize=13)
        axes[0, k].axis('off')
        axes[1, k].axis('off')
    headline(fig, 'Closest test/train pairs (duplicate check)', top=0.86)
    save_fig(fig, 'ds_duplicates.png')
else:
    print('No near duplicates found.')

## 2. Model

* **Backbone:** EfficientNetB0 with ImageNet weights (fallback: MobileNetV3Large if EfficientNetB0 cannot be loaded).
* **Input:** images are loaded at 256×256 and a **random 224×224 crop** is taken during training (centre crop when predicting).
* **Augmentation** (phone photos vary a lot): flips, rotation, brightness, contrast, random crop.
* **Class weights** because the classes have different sizes.
* **Two phases:** (1) only the new classification head is trained, the backbone is frozen; (2) the top layers of the backbone are unfrozen and fine-tuned with a small learning rate. **Early stopping** on the validation loss in both phases.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

keras.utils.set_random_seed(SEED)
print('TensorFlow', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))
if IN_COLAB and not tf.config.list_physical_devices('GPU'):
    print('WARNING: no GPU. Runtime → Change runtime type → T4 GPU.')

LOAD, IMG, BATCH = 256, 224, 32
AUTOTUNE = tf.data.AUTOTUNE


def load_image(path, size=LOAD):
    img = ImageOps.exif_transpose(Image.open(path)).convert('RGB').resize((size, size), Image.BILINEAR)
    return np.asarray(img, dtype=np.uint8)


def load_images(paths):
    with ThreadPoolExecutor(max_workers=8) as ex:
        return np.stack(list(ex.map(load_image, paths)))


t0 = time.time()
X_train, y_train = load_images(train.path), train.y.values.astype('int32')
X_val, y_val = load_images(val.path), val.y.values.astype('int32')
X_test, y_test = load_images(test.path), test.y.values.astype('int32')
print(f'Loaded {len(X_train) + len(X_val) + len(X_test)} images in {time.time() - t0:.0f} s', X_train.shape)


def make_ds(X, y, training):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if training:
        ds = ds.shuffle(2048, seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH).map(lambda a, b: (tf.cast(a, tf.float32), b), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)


train_ds, val_ds = make_ds(X_train, y_train, True), make_ds(X_val, y_val, False)

from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight('balanced', classes=np.arange(len(CLASSES)), y=y_train)
CLASS_WEIGHT = {i: float(w) for i, w in enumerate(cw)}
print('Class weights:', {CLASSES[i]: round(w, 2) for i, w in CLASS_WEIGHT.items()})

In [ ]:
def make_backbone():
    try:
        base = keras.applications.EfficientNetB0(include_top=False, weights='imagenet', input_shape=(IMG, IMG, 3))
        return base, 'EfficientNetB0'
    except Exception as e:  # for example no internet for the weights
        print('EfficientNetB0 could not be loaded, using MobileNetV3Large:', e)
        base = keras.applications.MobileNetV3Large(include_top=False, weights='imagenet', input_shape=(IMG, IMG, 3), include_preprocessing=True)
        return base, 'MobileNetV3Large'


@keras.utils.register_keras_serializable(package='agropulse')
class CropRandomOrCenter(layers.Layer):
    """Random crop while training, centre crop when predicting.
    (The built-in RandomCrop does not crop at prediction time in recent Keras versions.)"""

    def __init__(self, size, **kwargs):
        super().__init__(**kwargs)
        self.size = size
        self.random_crop = layers.RandomCrop(size, size)
        self.center_crop = layers.CenterCrop(size, size)

    def call(self, x, training=False):
        return self.random_crop(x, training=True) if training else self.center_crop(x)

    def compute_output_shape(self, input_shape):
        return (input_shape[0], self.size, self.size, input_shape[-1])

    def get_config(self):
        return {**super().get_config(), 'size': self.size}


base, BACKBONE = make_backbone()
base.trainable = False

augment = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.1),
    layers.RandomBrightness(0.2, value_range=(0, 255)),
    layers.RandomContrast(0.2),
], name='augment')

inputs = keras.Input((LOAD, LOAD, 3), name='image')
x = CropRandomOrCenter(IMG, name='crop')(inputs)   # random crop in training, centre crop when predicting
x = augment(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D(name='gap')(x)
x = layers.Dropout(0.3, name='drop')(x)
outputs = layers.Dense(len(CLASSES), activation='softmax', name='head')(x)
model = keras.Model(inputs, outputs, name='olive_leaf_model')
print('Backbone:', BACKBONE, '| parameters:', f'{model.count_params():,}')


def find_backbone(m):
    """The backbone inside a loaded model (the nested model that is not the augmentation block)."""
    return next(l for l in m.layers if isinstance(l, keras.Model) and not l.name.startswith('augment'))


def compile_model(lr):
    model.compile(optimizer=keras.optimizers.Adam(lr), loss='sparse_categorical_crossentropy', metrics=['accuracy'])


def early_stop(patience):
    return keras.callbacks.EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True, verbose=1)

### 2.1 Phase 1: train the new head (backbone frozen)

In [ ]:
V1_CKPT = CHECKPOINT_DIR / 'v1.keras'
TRAINED_V1 = RETRAIN or not V1_CKPT.exists()
if TRAINED_V1:
    compile_model(1e-3)
    t0 = time.time()
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=15, class_weight=CLASS_WEIGHT, callbacks=[early_stop(3)], verbose=2)
    print(f'Phase 1 took {time.time() - t0:.0f} s. Best validation accuracy: {max(h1.history["val_accuracy"]):.3f}')
else:
    print(f'Found {V1_CKPT.name} in the checkpoints folder: training of v1 is skipped and the saved model is loaded in the next cell. (RETRAIN = True in the setup cell trains again.)')

### 2.2 Phase 2: fine-tune the top layers of the backbone

The last 30 layers are unfrozen (BatchNorm layers stay frozen) and trained with a 30× smaller learning rate.

In [ ]:
N_FINE_TUNE = 30
if TRAINED_V1:
    base.trainable = True
    for layer in base.layers[:-N_FINE_TUNE]:
        layer.trainable = False
    for layer in base.layers[-N_FINE_TUNE:]:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    compile_model(3e-5)
    t0 = time.time()
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=20, class_weight=CLASS_WEIGHT, callbacks=[early_stop(4)], verbose=2)
    print(f'Phase 2 took {time.time() - t0:.0f} s. Best validation accuracy: {max(h2.history["val_accuracy"]):.3f}')
    model.save(V1_CKPT)   # so the next run can skip training
    print('Saved', V1_CKPT)
else:
    model = keras.models.load_model(V1_CKPT)
    base = find_backbone(model)
    print('Loaded v1 from', V1_CKPT)

In [ ]:
if not TRAINED_V1:
    print('v1 was loaded from a checkpoint, so there is no new training history. (training_history.png from the first run is still in the figures folder.)')
else:
    hist = pd.concat([pd.DataFrame(h1.history).assign(phase='1 · head only'), pd.DataFrame(h2.history).assign(phase='2 · fine-tune')], ignore_index=True)
    hist['epoch'] = np.arange(1, len(hist) + 1)
    switch = len(h1.history['loss']) + 0.5

    fig, (a1, a2) = plt.subplots(1, 2, figsize=FIGSIZE)
    a1.plot(hist.epoch, hist.loss, label='Training', color='#2E6FB0', lw=3)
    a1.plot(hist.epoch, hist.val_loss, label='Validation', color='#C9553A', lw=3)
    a1.set_title('Loss')
    a2.plot(hist.epoch, hist.accuracy, label='Training', color='#2E6FB0', lw=3)
    a2.plot(hist.epoch, hist.val_accuracy, label='Validation', color='#C9553A', lw=3)
    a2.set_title('Accuracy')
    for ax in (a1, a2):
        ax.axvline(switch, color='#6B6B6B', ls='--')
        ax.text(switch + 0.2, ax.get_ylim()[0] + (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.05, 'fine-tuning starts', color='#6B6B6B', fontsize=13)
        ax.set_xlabel('Epoch')
        ax.grid(color='#E6E6E2')
    a1.legend()
    headline(fig, 'Training history', top=0.84)
    save_fig(fig, 'training_history.png');

## 3. Evaluation

### 3.1 Official Denizli test set

The test set was not used for training or for choosing when to stop.

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score


def predict_probs(m, X):
    return np.concatenate([m.predict(X[i:i + 256].astype('float32'), verbose=0) for i in range(0, len(X), 256)])


def summarize(y_true, y_pred, labels):
    return {'images': int(len(y_true)), 'accuracy': accuracy_score(y_true, y_pred), 'macro_f1': f1_score(y_true, y_pred, labels=labels, average='macro', zero_division=0)}


probs_test = predict_probs(model, X_test)
pred_test = probs_test.argmax(1)
den_res = summarize(y_test, pred_test, list(range(len(CLASSES))))
print(f'Denizli test: accuracy {den_res["accuracy"]:.3f} · macro F1 {den_res["macro_f1"]:.3f} · {den_res["images"]} images')
print()
report = classification_report(y_test, pred_test, target_names=CLASSES, digits=3, output_dict=True, zero_division=0)
per_class = pd.DataFrame({k: v for k, v in report.items() if isinstance(v, dict)}).T.loc[CLASSES + ['macro avg'], ['precision', 'recall', 'f1-score', 'support']]
per_class['support'] = per_class['support'].astype(int)
display(per_class.round(3))

# The same score without the test images that have a near duplicate in train.
clean = ~test.near_dup.values
den_clean = summarize(y_test[clean], pred_test[clean], list(range(len(CLASSES))))
print(f'Without the {int((~clean).sum())} test images that have a near duplicate in train: accuracy {den_clean["accuracy"]:.3f} · macro F1 {den_clean["macro_f1"]:.3f} · {den_clean["images"]} images')

In [ ]:
def plot_confusion(cm, labels, title, name, normalize=True):
    fig, ax = plt.subplots(figsize=FIGSIZE)
    shown = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1) if normalize else cm
    im = ax.imshow(shown, cmap='Greens', vmin=0, vmax=1 if normalize else None)
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, fontsize=15)
    ax.set_yticklabels(labels, fontsize=15)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f'{cm[i, j]}\n{shown[i, j] * 100:.0f} %', ha='center', va='center', fontsize=20, fontweight='bold',
                    color='white' if shown[i, j] > 0.55 else '#151515')
    ax.grid(False)
    fig.colorbar(im, ax=ax, pad=0.02).set_label('Share of the true class' if normalize else 'Images')
    headline(fig, title, top=0.84)
    return save_fig(fig, name)


cm_test = confusion_matrix(y_test, pred_test, labels=list(range(len(CLASSES))))
plot_confusion(cm_test, CLASSES, f'Denizli test set: confusion matrix (accuracy {den_res["accuracy"] * 100:.1f} %)', 'confusion_denizli_test.png');

### 3.2 External test 1: Edincik (different place, white background, light box)

The Edincik photos have only two classes. We map *healthy → Healthy* and *peacock spot → Olive Peacock Spot* and evaluate on **these two classes only**: for each image the model has to choose between these two (the *Aculus Olearius* score is ignored). For comparison we also show the score on the Denizli test images of the same two classes.

In [ ]:
HP = [CLASS_ID['Healthy'], CLASS_ID['Olive Peacock Spot']]


def two_class_pred(probs):
    """Choose between Healthy and Olive Peacock Spot only. Returns 0 = Healthy, 1 = Peacock spot."""
    return probs[:, HP].argmax(1)


def two_class_truth(y):
    return (y == CLASS_ID['Olive Peacock Spot']).astype(int)


# Denizli test, same two classes.
m2 = np.isin(y_test, HP)
den2_res = summarize(two_class_truth(y_test[m2]), two_class_pred(probs_test[m2]), [0, 1])

if len(edi):
    X_edi = load_images(edi.path)
    y_edi = edi.y.values.astype('int32')
    probs_edi = predict_probs(model, X_edi)
    edi_res = summarize(two_class_truth(y_edi), two_class_pred(probs_edi), [0, 1])
    edi_res3 = accuracy_score(y_edi, probs_edi.argmax(1))
    print(f'Edincik (2 classes): accuracy {edi_res["accuracy"]:.3f} · macro F1 {edi_res["macro_f1"]:.3f} · {edi_res["images"]} images')
    print(f'Edincik if all 3 classes are allowed (a Aculus prediction counts as wrong): accuracy {edi_res3:.3f}')
    cm_edi = confusion_matrix(two_class_truth(y_edi), two_class_pred(probs_edi), labels=[0, 1])
    plot_confusion(cm_edi, ['Healthy', 'Peacock spot'], f'Edincik external test: confusion matrix (accuracy {edi_res["accuracy"] * 100:.1f} %)', 'confusion_edincik.png');
else:
    edi_res = None
    print('Edincik not available: skipped.')

### 3.3 External test 2: Greek field photos

If you have phone photos taken in Greece, put them in `MyDrive/AgroPulse/data/field_photos/<class>/` (for example `healthy/`, `peacock_spot/`, `aculus_olearius/`). Folders with other names are mapped automatically when the name contains *healthy*, *peacock* or *aculus*.

In [ ]:
field_res = None
if FIELD_PHOTOS.exists():
    fp = scan_images(FIELD_PHOTOS)
    fp['label'] = fp.class_raw.map(normalize_class)
    fp = fp[fp.label.notna()].copy()
    fp['y'] = fp.label.map(CLASS_ID)
else:
    fp = pd.DataFrame()

if len(fp):
    X_fp, y_fp = load_images(fp.path), fp.y.values.astype('int32')
    probs_fp = predict_probs(model, X_fp)
    present = sorted(set(y_fp.tolist()))
    field_res = summarize(y_fp, probs_fp.argmax(1), present)
    print(f'Greek field photos: {field_res["images"]} images in {len(present)} classes · accuracy {field_res["accuracy"]:.3f} · macro F1 {field_res["macro_f1"]:.3f}')
    print(fp.label.value_counts().to_string())
    cm_fp = confusion_matrix(y_fp, probs_fp.argmax(1), labels=list(range(len(CLASSES))))
    plot_confusion(cm_fp, CLASSES, 'Greek field photos: confusion matrix', 'confusion_field_photos.png');
else:
    print(f'No field photos found in {FIELD_PHOTOS}. Skipped. (Add phone photos from Greece there and run this cell again.)')

### 3.4 One table: how well does it work where?

In [ ]:
rows = [('Denizli test (official, 3 classes)', den_res)]
rows.append(('Denizli test, only Healthy vs Peacock spot', den2_res))
if edi_res:
    rows.append(('Edincik (external, Healthy vs Peacock spot)', edi_res))
if field_res:
    rows.append(('Greek field photos (external)', field_res))
results_df = pd.DataFrame([{'Test set': n, 'Images': r['images'], 'Accuracy': f'{r["accuracy"] * 100:.1f} %', 'Macro F1': f'{r["macro_f1"]:.3f}'} for n, r in rows])
display(results_df)

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.axis('off')
tbl = ax.table(cellText=results_df.values.tolist(), colLabels=list(results_df.columns), loc='center', cellLoc='center', colWidths=[0.46, 0.14, 0.18, 0.18])
tbl.auto_set_font_size(False)
tbl.set_fontsize(17)
tbl.scale(1, 3.4)
for (r, c), cell in tbl.get_celld().items():
    cell.set_edgecolor('#E6E6E2')
    if r == 0:
        cell.set_facecolor('#151515')
        cell.set_text_props(color='white', fontweight='bold')
    elif c == 0:
        cell.set_text_props(ha='left')
        cell._loc = 'left'
headline(fig, 'Results on three kinds of photos', top=0.84)
save_fig(fig, 'results_table.png');

In [ ]:
from IPython.display import Markdown, display

lines = []
if edi_res:
    drop = (den2_res['accuracy'] - edi_res['accuracy']) * 100
    lines.append(f'* **Edincik vs Denizli (same two classes):** accuracy goes from {den2_res["accuracy"] * 100:.1f} % to {edi_res["accuracy"] * 100:.1f} % ({drop:+.1f} points).')
if field_res:
    drop = (den_res['accuracy'] - field_res['accuracy']) * 100
    lines.append(f'* **Greek field photos vs Denizli test:** accuracy goes from {den_res["accuracy"] * 100:.1f} % to {field_res["accuracy"] * 100:.1f} % ({-drop:+.1f} points).')
display(Markdown('\n'.join(lines) if lines else '*Only the Denizli test set was available.*'))

**Why the scores differ between test sets, in plain English**

* **The Denizli test photos look like the Denizli training photos:** same collection, same cameras, one leaf per photo on a plain surface (white or grey paper, a wooden table), taken in the same period. The model learns the leaves *and* a bit of the "look" of these photos, so this is the easiest test and the best-case score. The duplicate check in section 1.5 shows that some test photos are even the **same leaf** as a training photo, which makes this score a little optimistic (the score without them is printed in section 3.1).
* **Background:** both datasets show a single leaf on a plain background. **Edincik** photos were taken in a **light box on white**: very clean, even light and no shadows. **Real field photos** (a leaf on the tree, in a hand, with soil, sky, other leaves and shadows) look very different. This is usually the biggest gap, and it is what the Greek field-photo test is meant to measure.
* **Light and camera:** sun, shade, flash, phone model and focus change colours and sharpness. A leaf that looks yellow-green in strong sun can look dull in shade.
* **Region and season:** olive varieties, leaf size and disease stage differ between Denizli, Balıkesir and Greece, and the training photos are from one season.
* **Honest reading:** the lower numbers on external sets are the more realistic ones for a farmer in the field. The goal of the Greek field photos is to measure exactly this gap.

### 3.5 Does the model look at the spots? (Grad-CAM)

Grad-CAM shows which parts of the image make the model choose a class (red = strongest). For a good model the heat should sit **on the spots or damaged areas, not on the background**. Two images per class, taken from correctly classified test images.

In [ ]:
LAST_FEATURES = keras.Model(base.input, base.output)   # last feature maps of the backbone
GAP, HEAD = model.get_layer('gap'), model.get_layer('head')
CENTER = layers.CenterCrop(IMG, IMG)


def grad_cam(img_uint8, class_id=None):
    x = CENTER(tf.cast(img_uint8[None], tf.float32))
    with tf.GradientTape() as tape:
        feats = LAST_FEATURES(x, training=False)
        tape.watch(feats)
        preds = HEAD(GAP(feats))
        cid = int(tf.argmax(preds[0])) if class_id is None else class_id
        score = preds[:, cid]
    grads = tape.gradient(score, feats)
    weights = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    cam = tf.nn.relu(tf.reduce_sum(weights * feats, axis=-1))[0].numpy()
    cam = cam / (cam.max() + 1e-8)
    cam = np.asarray(Image.fromarray((cam * 255).astype('uint8')).resize((IMG, IMG), Image.BILINEAR)) / 255.0
    return x[0].numpy().astype('uint8'), cam, cid


def overlay(img, cam):
    heat = (mpl.colormaps['jet'](cam)[..., :3] * 255).astype('uint8')
    return (0.55 * img + 0.45 * heat).astype('uint8')


rng = np.random.default_rng(SEED)
picks = []
for c in range(len(CLASSES)):
    ok = np.where((y_test == c) & (pred_test == c))[0]
    picks.append(rng.choice(ok, size=min(2, len(ok)), replace=False) if len(ok) else [])

fig, axes = plt.subplots(len(CLASSES), 4, figsize=FIGSIZE)
for r, c in enumerate(range(len(CLASSES))):
    for k in range(2):
        a_img, a_cam = axes[r, 2 * k], axes[r, 2 * k + 1]
        a_img.axis('off')
        a_cam.axis('off')
        if k < len(picks[r]):
            i = int(picks[r][k])
            img, cam, cid = grad_cam(X_test[i])
            a_img.imshow(img)
            a_cam.imshow(overlay(img, cam))
            conf = probs_test[i, cid]
            a_img.set_title(f'{CLASSES[c]}', fontsize=14, loc='left')
            a_cam.set_title(f'model attention · {conf * 100:.0f} % sure', fontsize=13, loc='left')
headline(fig, 'Grad-CAM: where does the model look?', top=0.82)
fig.subplots_adjust(hspace=0.2, wspace=0.04, left=0.02, right=0.98, bottom=0.02)
save_fig(fig, 'gradcam.png');

### 3.6 Example predictions (including the worst mistakes)

Left: four random correct predictions. Right: the four most **confident mistakes**.

In [ ]:
conf_test = probs_test.max(1)
wrong = np.where(pred_test != y_test)[0]
right = np.where(pred_test == y_test)[0]
worst = wrong[np.argsort(-conf_test[wrong])][:4]
rand_ok = rng.choice(right, size=4, replace=False)
show_ids = list(rand_ok) + list(worst)

fig, axes = plt.subplots(2, 4, figsize=FIGSIZE)
for ax, i in zip(axes.ravel(), show_ids):
    ax.imshow(X_test[i])
    ok = pred_test[i] == y_test[i]
    ax.set_title(f'True: {CLASSES[y_test[i]]}\nPredicted: {CLASSES[pred_test[i]]} ({conf_test[i] * 100:.0f} %)', fontsize=12, color='#3F6B2F' if ok else '#C9553A')
    ax.axis('off')
headline(fig, 'Example predictions: 4 correct (top) and the 4 most confident mistakes (bottom)', top=0.86)
fig.subplots_adjust(hspace=0.35, wspace=0.05, left=0.02, right=0.98, bottom=0.02)
save_fig(fig, 'examples.png');
print(f'The model made {len(wrong)} mistakes on {len(y_test)} test images.')

### 3.7 Confidence

If we only trust predictions where the model is at least 60 % sure, how many images are left and how accurate are they?

In [ ]:
THRESHOLD = 0.6


def confidence_stats(probs, y_true, pred=None):
    pred = probs.argmax(1) if pred is None else pred
    conf = probs.max(1)
    sure = conf >= THRESHOLD
    return {'share_sure': float(sure.mean()), 'acc_sure': float((pred[sure] == y_true[sure]).mean()) if sure.any() else float('nan'), 'acc_unsure': float((pred[~sure] == y_true[~sure]).mean()) if (~sure).any() else float('nan')}


conf_rows = [('Denizli test', confidence_stats(probs_test, y_test))]
if edi_res:
    p2 = probs_edi[:, HP] / probs_edi[:, HP].sum(1, keepdims=True)
    conf_rows.append(('Edincik (2 classes)', confidence_stats(p2, two_class_truth(y_edi))))
if field_res:
    conf_rows.append(('Greek field photos', confidence_stats(probs_fp, y_fp)))
conf_df = pd.DataFrame([{'Test set': n, f'Images with confidence ≥ {THRESHOLD:.0%}': f'{s["share_sure"] * 100:.1f} %',
                         'Accuracy on those': f'{s["acc_sure"] * 100:.1f} %', 'Accuracy on the rest': f'{s["acc_unsure"] * 100:.1f} %' if s['acc_unsure'] == s['acc_unsure'] else '–'} for n, s in conf_rows])
display(conf_df)

## 3.8 Experiment: v2 (letterbox + stronger colour augmentation)

Everything above is **v1** and stays exactly as it is, for comparison. v2 changes only two things, trained with the **same split, the same seed and the same hyper-parameters**:

1. **Letterbox instead of squashing.** Each image is padded to a square with **its own border colour** (median colour of the outer edge) and then resized to 224×224. The leaf keeps its shape: the long, thin Edincik leaves are no longer stretched, and there is no centre crop that could cut off a leaf tip. The same function is used in training and in every evaluation.
2. **Stronger colour augmentation** (training only): brightness ±30 %, contrast ±30 %, saturation ±30 %, a small hue shift (±4 %), slight blur (30 % of the images), and **random grey backgrounds** (50 % of the images). The background is found as "pixels close to the border colour", which works for plain paper, wood and white boxes but is not a perfect leaf mask. Flip, rotation (±10 %) and zoom (±10 %) are done inside the model.

**No Edincik image is used for training or for tuning v2.** Edincik is only used afterwards, to decide between v1 and v2 (see the rule at the end of this section).

In [ ]:
# None = decide with the rule in 3.8.3. 'v1' or 'v2' = export that version, whatever the rule says (a deliberate choice, written in the model card).
FORCE_VERSION = 'v2'

V1 = {'model': model, 'base': base, 'probs_test': probs_test}   # the first run, kept for the comparison

BG_TOL = 60   # a pixel counts as background when |R,G,B difference| to the border colour sums to less than this


def letterbox(path, size=IMG):
    """Pad to a square with the image's own border colour, resize to size x size. Returns (image, background mask)."""
    a = np.asarray(ImageOps.exif_transpose(Image.open(path)).convert('RGB'))
    h, w = a.shape[:2]
    ring = max(2, int(0.02 * min(h, w)))
    border = np.concatenate([a[:ring].reshape(-1, 3), a[-ring:].reshape(-1, 3), a[:, :ring].reshape(-1, 3), a[:, -ring:].reshape(-1, 3)])
    colour = np.median(border, axis=0).astype(np.uint8)
    side = max(h, w)
    canvas = np.empty((side, side, 3), dtype=np.uint8)
    canvas[:] = colour
    top, left = (side - h) // 2, (side - w) // 2
    canvas[top:top + h, left:left + w] = a
    out = np.asarray(Image.fromarray(canvas).resize((size, size), Image.BILINEAR), dtype=np.uint8)
    diff = np.abs(out.astype(np.int16) - colour.astype(np.int16)).sum(-1)
    return out, diff < BG_TOL


def load_letterboxed(paths):
    with ThreadPoolExecutor(max_workers=8) as ex:
        res = list(ex.map(letterbox, list(paths)))
    return np.stack([r[0] for r in res]), np.stack([r[1] for r in res])


t0 = time.time()
Xl_train, Ml_train = load_letterboxed(train.path)
Xl_val, _ = load_letterboxed(val.path)
Xl_test, _ = load_letterboxed(test.path)
print(f'Letterboxed {len(Xl_train) + len(Xl_val) + len(Xl_test)} images in {time.time() - t0:.0f} s', Xl_train.shape)


def tf_blur(img, sigma):
    ax = tf.range(-2, 3, dtype=tf.float32)
    g = tf.exp(-(ax ** 2) / (2.0 * sigma ** 2))
    g = g / tf.reduce_sum(g)
    kern = tf.tile((g[:, None] * g[None, :])[:, :, None, None], [1, 1, 3, 1])
    return tf.nn.depthwise_conv2d(img[None], kern, [1, 1, 1, 1], 'SAME')[0]


def colour_aug(img, mask):
    """Stronger colour augmentation for one image (0..255 float in, 0..255 float out)."""
    img = tf.cast(img, tf.float32)
    grey = tf.random.uniform([], 60.0, 210.0)
    grey_bg = tf.where(mask[..., None], grey + tf.random.normal(tf.shape(img), stddev=4.0), img)
    img = tf.where(tf.random.uniform([]) < 0.5, grey_bg, img)
    img = tf.image.random_saturation(img / 255.0, 0.7, 1.3)
    img = tf.image.random_hue(img, 0.04) * 255.0
    img = img * tf.random.uniform([], 0.7, 1.3)
    mean = tf.reduce_mean(img)
    img = (img - mean) * tf.random.uniform([], 0.7, 1.3) + mean
    img = tf.where(tf.random.uniform([]) < 0.3, tf_blur(img, tf.random.uniform([], 0.5, 1.2)), img)
    return tf.clip_by_value(img, 0.0, 255.0)


def make_ds_v2(X, M, y, training):
    if not training:
        return tf.data.Dataset.from_tensor_slices((X, y)).batch(BATCH).map(lambda a, b: (tf.cast(a, tf.float32), b), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
    ds = tf.data.Dataset.from_tensor_slices((X, M, y)).shuffle(2048, seed=SEED, reshuffle_each_iteration=True)
    return ds.map(lambda a, m, b: (colour_aug(a, m), b), num_parallel_calls=AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)


train_ds2, val_ds2 = make_ds_v2(Xl_train, Ml_train, y_train, True), make_ds_v2(Xl_val, None, y_val, False)

# Look at what the model sees: v1 (squashed) | v2 letterbox | three augmented versions.
examples = []
if len(edi):
    examples += edi.sample(2, random_state=SEED).path.tolist()
examples += train.sample(4 - len(examples), random_state=SEED).path.tolist()
fig, axes = plt.subplots(len(examples), 5, figsize=FIGSIZE)
for r, p in enumerate(examples):
    lb, mk = letterbox(p)
    shown = [load_image(p), lb] + [colour_aug(lb, mk).numpy().astype('uint8') for _ in range(3)]
    for k, im in enumerate(shown):
        axes[r, k].imshow(im)
        axes[r, k].axis('off')
for k, t in enumerate(['v1: squashed to a square', 'v2: letterbox', 'v2 augmented', 'v2 augmented', 'v2 augmented']):
    axes[0, k].set_title(t, fontsize=15)
headline(fig, 'Preprocessing: v1 vs v2', top=0.86)
fig.subplots_adjust(hspace=0.05, wspace=0.04, left=0.02, right=0.98, bottom=0.02)
save_fig(fig, 'v2_preprocessing_edincik_photos.png');

### 3.8.1 Train v2

Same recipe as v1: phase 1 (head only, learning rate 1e-3, up to 15 epochs, patience 3), phase 2 (last 30 layers, learning rate 3e-5, up to 20 epochs, patience 4), same class weights, same validation images.

In [ ]:
V2_CKPT = CHECKPOINT_DIR / 'v2.keras'
TRAINED_V2 = RETRAIN or not V2_CKPT.exists()
if TRAINED_V2:
    keras.utils.set_random_seed(SEED)
    base2, BACKBONE2 = make_backbone()
    base2.trainable = False

    augment2 = keras.Sequential([
        layers.RandomFlip('horizontal_and_vertical'),
        layers.RandomRotation(0.1, fill_mode='nearest'),
        layers.RandomZoom(0.1, fill_mode='nearest'),
    ], name='augment_v2')

    in2 = keras.Input((IMG, IMG, 3), name='image')
    z2 = augment2(in2)
    z2 = base2(z2, training=False)
    z2 = layers.GlobalAveragePooling2D(name='gap')(z2)
    z2 = layers.Dropout(0.3, name='drop')(z2)
    out2 = layers.Dense(len(CLASSES), activation='softmax', name='head')(z2)
    model2 = keras.Model(in2, out2, name='olive_leaf_model_v2')


    def compile2(lr):
        model2.compile(optimizer=keras.optimizers.Adam(lr), loss='sparse_categorical_crossentropy', metrics=['accuracy'])


    t0 = time.time()
    compile2(1e-3)
    g1 = model2.fit(train_ds2, validation_data=val_ds2, epochs=15, class_weight=CLASS_WEIGHT, callbacks=[early_stop(3)], verbose=2)
    print(f'v2 phase 1 took {time.time() - t0:.0f} s. Best validation accuracy: {max(g1.history["val_accuracy"]):.3f}')

    base2.trainable = True
    for layer in base2.layers[:-N_FINE_TUNE]:
        layer.trainable = False
    for layer in base2.layers[-N_FINE_TUNE:]:
        if isinstance(layer, layers.BatchNormalization):
            layer.trainable = False
    t0 = time.time()
    compile2(3e-5)
    g2 = model2.fit(train_ds2, validation_data=val_ds2, epochs=20, class_weight=CLASS_WEIGHT, callbacks=[early_stop(4)], verbose=2)
    print(f'v2 phase 2 took {time.time() - t0:.0f} s. Best validation accuracy: {max(g2.history["val_accuracy"]):.3f}')
    if TRAINED_V1:
        print(f'v1 best validation accuracy for comparison: {max(max(h1.history["val_accuracy"]), max(h2.history["val_accuracy"])):.3f}')
    model2.save(V2_CKPT)   # the v2 weights stay on Drive: a later run can export without training
    print('Saved', V2_CKPT)
else:
    model2 = keras.models.load_model(V2_CKPT)
    base2 = find_backbone(model2)
    print('Loaded v2 from', V2_CKPT, '(training skipped)')

### 3.8.2 v1 against v2

Same test images, same metrics. *Denizli de-duplicated* leaves out the test images that have a near duplicate in train (section 1.5). *Edincik* is the 2-class test of section 3.2 (the model chooses between Healthy and Peacock spot). Recall = the share of the true leaves of that class that the model finds.

In [ ]:
probs_test2 = predict_probs(model2, Xl_test)
pred_test2 = probs_test2.argmax(1)
ALL3 = list(range(len(CLASSES)))


def metrics_denizli(probs):
    pred = probs.argmax(1)
    full, dedup = summarize(y_test, pred, ALL3), summarize(y_test[clean], pred[clean], ALL3)
    return {'Denizli test: accuracy': full['accuracy'], 'Denizli test: macro F1': full['macro_f1'],
            'Denizli de-duplicated: accuracy': dedup['accuracy'], 'Denizli de-duplicated: macro F1': dedup['macro_f1']}


def metrics_edincik(probs):
    truth, pred = two_class_truth(y_edi), two_class_pred(probs)
    cm = confusion_matrix(truth, pred, labels=[0, 1])
    s = summarize(truth, pred, [0, 1])
    return {'Edincik: accuracy': s['accuracy'], 'Edincik: macro F1': s['macro_f1'],
            'Edincik: healthy recall': cm[0, 0] / max(cm[0].sum(), 1), 'Edincik: peacock spot recall': cm[1, 1] / max(cm[1].sum(), 1)}, cm


m1, m2v = metrics_denizli(probs_test), metrics_denizli(probs_test2)
cm_edi2 = None
if len(edi):
    Xl_edi, _ = load_letterboxed(edi.path)
    probs_edi2 = predict_probs(model2, Xl_edi)
    e1, _ = metrics_edincik(probs_edi)
    e2, cm_edi2 = metrics_edincik(probs_edi2)
    m1.update(e1)
    m2v.update(e2)

cmp_df = pd.DataFrame({'Metric': list(m1), 'v1': [m1[k] for k in m1], 'v2': [m2v[k] for k in m1]})
cmp_df['Change'] = [f'{b - a:+.3f}' if 'F1' in k else f'{(b - a) * 100:+.1f} pts' for k, a, b in zip(cmp_df.Metric, cmp_df.v1, cmp_df.v2)]
for col in ('v1', 'v2'):
    cmp_df[col] = [f'{v:.3f}' if 'F1' in k else f'{v * 100:.1f} %' for k, v in zip(cmp_df.Metric, cmp_df[col])]
display(cmp_df)

fig, ax = plt.subplots(figsize=FIGSIZE)
ax.axis('off')
tbl = ax.table(cellText=cmp_df.values.tolist(), colLabels=list(cmp_df.columns), loc='center', cellLoc='center', colWidths=[0.46, 0.17, 0.17, 0.17])
tbl.auto_set_font_size(False)
tbl.set_fontsize(16)
tbl.scale(1, 2.6)
for (r, c), cell in tbl.get_celld().items():
    cell.set_edgecolor('#E6E6E2')
    if r == 0:
        cell.set_facecolor('#151515')
        cell.set_text_props(color='white', fontweight='bold')
    elif c == 0:
        cell.set_text_props(ha='left')
        cell._loc = 'left'
headline(fig, 'v1 (squashed, mild augmentation) vs v2 (letterbox, strong colour augmentation)', top=0.84)
save_fig(fig, 'v1_vs_v2.png')

if cm_edi2 is not None:
    plot_confusion(cm_edi2, ['Healthy', 'Peacock spot'], f'Edincik confusion matrix, v2 (accuracy {m2v["Edincik: accuracy"] * 100:.1f} %)', 'confusion_edincik_v2.png')

### 3.8.3 Which version is the final model?

**Rule:** keep v2 only if it is better on Edincik (higher macro F1) **and** its accuracy on the official Denizli test set stays at or above **94 %**. Otherwise keep v1. If Edincik is not available, the rule cannot be applied and v1 is kept.

**Override:** `FORCE_VERSION` at the top of section 3.8. When it is `'v1'` or `'v2'`, that version is exported and the gate is skipped. The model card then states the rule, what it said, and why it was overridden.

Note: Edincik was not used for training, but it is used here to choose between two versions, so it is no longer a completely untouched test. The Greek field photos (if you have them) remain the cleaner final check.

In [ ]:
DENIZLI_GATE = 0.94
if cm_edi2 is None:
    RULE_FINAL, RULE_REASON = 'v1', 'Edincik not available, so v2 could not be judged: v1 kept.'
elif m2v['Edincik: macro F1'] > m1['Edincik: macro F1'] and m2v['Denizli test: accuracy'] >= DENIZLI_GATE:
    RULE_FINAL = 'v2'
    RULE_REASON = (f'v2 is better on Edincik (macro F1 {m1["Edincik: macro F1"]:.3f} to {m2v["Edincik: macro F1"]:.3f}) and keeps Denizli accuracy at '
                   f'{m2v["Denizli test: accuracy"] * 100:.1f} % (gate: {DENIZLI_GATE:.0%}).')
elif m2v['Edincik: macro F1'] > m1['Edincik: macro F1']:
    RULE_FINAL = 'v1'
    RULE_REASON = f'v2 is better on Edincik but its Denizli accuracy is {m2v["Denizli test: accuracy"] * 100:.1f} %, below the {DENIZLI_GATE:.0%} gate: v1 kept.'
else:
    RULE_FINAL = 'v1'
    RULE_REASON = f'v2 is not better on Edincik (macro F1 {m1["Edincik: macro F1"]:.3f} vs {m2v["Edincik: macro F1"]:.3f}): v1 kept.'

if FORCE_VERSION in ('v1', 'v2'):
    FINAL = FORCE_VERSION
    OVERRIDE = FINAL != RULE_FINAL
else:
    FINAL, OVERRIDE = RULE_FINAL, False

# The reason text only claims better generalisation when the numbers of this run show it.
EDI_GAIN = cm_edi2 is not None and m2v['Edincik: macro F1'] > m1['Edincik: macro F1']
if OVERRIDE and FINAL == 'v2' and EDI_GAIN:
    SELECTION_REASON = (f'The rule chose {RULE_FINAL} ({RULE_REASON}) but we exported {FINAL} on purpose. Reason: {FINAL} generalises much better to a new region '
                        f'(Edincik macro F1 {m2v["Edincik: macro F1"]:.3f} against {m1["Edincik: macro F1"]:.3f}), which is closer to real use on farmers phones; '
                        f'the cost is {abs(m2v["Denizli test: accuracy"] - m1["Denizli test: accuracy"]) * 100:.1f} points of Denizli accuracy.')
elif OVERRIDE:
    SELECTION_REASON = f'The rule chose {RULE_FINAL} ({RULE_REASON}) but we exported {FINAL} on purpose.'
else:
    SELECTION_REASON = RULE_REASON

if FINAL == 'v2':
    final_model, final_base, final_X_test, final_probs_test = model2, base2, Xl_test, probs_test2
    PREPROCESS = 'letterbox'
    PREPROCESS_TEXT = 'letterbox: padded to a square with the photo own border colour, resized to 224×224 (no crop)'
else:
    final_model, final_base, final_X_test, final_probs_test = model, base, X_test, probs_test
    PREPROCESS = 'resize_center_crop'
    PREPROCESS_TEXT = 'resized (squashed) to 256×256, centre crop 224×224'
final_pred_test = final_probs_test.argmax(1)
print(f'FINAL MODEL: {FINAL}' + (' (override)' if OVERRIDE else ''))
print(SELECTION_REASON)

## 4. Baseline: a general vision language model (optional)

How does a general-purpose vision LLM do on the same task? We send **60 Denizli test images (20 per class)** with a fixed prompt and the same three classes.
This cell only runs if an `ANTHROPIC_API_KEY` is available: set it as an environment variable, or in Colab as a *Secret* (key icon in the left bar) named `ANTHROPIC_API_KEY` with notebook access switched on.

In [ ]:
api_key = os.environ.get('ANTHROPIC_API_KEY')
if not api_key and IN_COLAB:
    try:
        from google.colab import userdata
        api_key = userdata.get('ANTHROPIC_API_KEY')
    except Exception:
        api_key = None

llm_df = None
if not api_key:
    print('ANTHROPIC_API_KEY not set: baseline skipped.')
else:
    import base64, io, subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'anthropic'], check=True)
    import anthropic

    LLM_MODEL = 'claude-sonnet-5-5'   # change here to try another model
    client = anthropic.Anthropic(api_key=api_key)
    PROMPT = ('This is a photo of an olive (Olea europaea) leaf. Classify it into exactly one of these classes: '
              '"Healthy", "Olive Peacock Spot" (round dark spots with a yellow halo, caused by Spilocaea oleaginea), '
              '"Aculus Olearius" (damage by the olive bud mite: yellowing, bronzing, deformed leaf). '
              'Answer with only the class name.')
    subset = pd.concat([test[test.label == c].sample(20, random_state=SEED) for c in CLASSES])
    sub_idx = subset.index.tolist()   # positions in `test` (it has a plain 0..n index)
    llm_pred = []
    for path in subset.path:
        img = ImageOps.exif_transpose(Image.open(path)).convert('RGB')
        img.thumbnail((1024, 1024))
        buf = io.BytesIO()
        img.save(buf, 'JPEG', quality=90)
        msg = client.messages.create(model=LLM_MODEL, max_tokens=20, messages=[{'role': 'user', 'content': [
            {'type': 'image', 'source': {'type': 'base64', 'media_type': 'image/jpeg', 'data': base64.b64encode(buf.getvalue()).decode()}},
            {'type': 'text', 'text': PROMPT}]}])
        answer = msg.content[0].text.strip().lower()
        pred = next((CLASS_ID[c] for c in CLASSES if c.lower() in answer), -1)   # -1 = not understood, counts as wrong
        llm_pred.append(pred)
    y_sub = subset.y.values
    llm_res = summarize(y_sub, np.array(llm_pred), list(range(len(CLASSES))))
    ours = summarize(y_sub, final_pred_test[sub_idx], list(range(len(CLASSES))))
    llm_df = pd.DataFrame([
        {'Model': f'Our model ({FINAL}, EfficientNetB0 fine-tuned)', 'Images': ours['images'], 'Accuracy': f'{ours["accuracy"] * 100:.1f} %', 'Macro F1': f'{ours["macro_f1"]:.3f}'},
        {'Model': f'Vision LLM ({LLM_MODEL}), no training', 'Images': llm_res['images'], 'Accuracy': f'{llm_res["accuracy"] * 100:.1f} %', 'Macro F1': f'{llm_res["macro_f1"]:.3f}'},
    ])
    display(llm_df)
    fig, ax = plt.subplots(figsize=FIGSIZE)
    ax.axis('off')
    tbl = ax.table(cellText=llm_df.values.tolist(), colLabels=list(llm_df.columns), loc='center', cellLoc='center', colWidths=[0.5, 0.15, 0.17, 0.17])
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(17)
    tbl.scale(1, 3.4)
    for (r, c), cell in tbl.get_celld().items():
        cell.set_edgecolor('#E6E6E2')
        if r == 0:
            cell.set_facecolor('#151515')
            cell.set_text_props(color='white', fontweight='bold')
    headline(fig, 'Our model against a general vision LLM (60 test images)', top=0.84)
    save_fig(fig, 'baseline_llm.png')

## 5. Export

The **final model** (v1 or v2, chosen in section 3.8.3) is exported:

* `research/models/leaf_disease.keras` (full Keras model) and `leaf_disease.tflite` (small, for the Hugging Face demo and later for the phone app).
* The TFLite model takes one RGB image with values 0–255 (float32) and returns the 3 class probabilities. **v1:** 256×256, squashed; the model does the centre crop itself. **v2:** 224×224 after the letterbox step, which the caller must do (the Hugging Face app does).
* `labels.json` lists the classes in the order of the output. `model_info.json` says which preprocessing and input size the model needs.

In [ ]:
# Inference-only model: (v1: centre crop) + backbone + head, no random layers, which converts cleanly to TFLite.
in_size = int(final_X_test.shape[1])
inf_in = keras.Input((in_size, in_size, 3), name='image')
z = layers.CenterCrop(IMG, IMG)(inf_in) if FINAL == 'v1' else inf_in
z = final_base(z, training=False)
z = final_model.get_layer('gap')(z)
inf_out = final_model.get_layer('head')(z)
inference_model = keras.Model(inf_in, inf_out, name='olive_leaf_inference')

final_model.save(MODEL_DIR / 'leaf_disease.keras')
try:
    converter = tf.lite.TFLiteConverter.from_keras_model(inference_model)
except Exception as e:  # some Keras versions need the SavedModel route
    print('from_keras_model failed, using a SavedModel:', e)
    inference_model.export('/tmp/olive_saved_model')
    converter = tf.lite.TFLiteConverter.from_saved_model('/tmp/olive_saved_model')
converter.optimizations = [tf.lite.Optimize.DEFAULT]   # smaller file (dynamic range quantization)
tflite_bytes = converter.convert()
(MODEL_DIR / 'leaf_disease.tflite').write_bytes(tflite_bytes)
(MODEL_DIR / 'labels.json').write_text(json.dumps(CLASSES, indent=2))
(MODEL_DIR / 'model_info.json').write_text(json.dumps({'version': FINAL, 'preprocess': PREPROCESS, 'input_size': in_size, 'value_range': [0, 255]}, indent=2))

# Check that the TFLite model gives the same answers as the Keras model.
interp = tf.lite.Interpreter(model_content=tflite_bytes)
interp.allocate_tensors()
inp, out = interp.get_input_details()[0], interp.get_output_details()[0]
lite_pred = []
for i in range(len(final_X_test)):
    interp.set_tensor(inp['index'], final_X_test[i:i + 1].astype('float32'))
    interp.invoke()
    lite_pred.append(int(interp.get_tensor(out['index'])[0].argmax()))
lite_pred = np.array(lite_pred)
n_test = len(final_X_test)
keras_acc_full = float((final_pred_test == y_test).mean())
lite_acc_full = float((lite_pred == y_test).mean())
agree_full = float((lite_pred == final_pred_test).mean())
print(f'Full Denizli test set ({n_test} images), final model {FINAL}:')
print(f'  Keras  accuracy {keras_acc_full * 100:.1f} %')
print(f'  TFLite accuracy {lite_acc_full * 100:.1f} %')
print(f'  Keras and TFLite give the same prediction on {agree_full * 100:.1f} % ({int((lite_pred == final_pred_test).sum())} of {n_test})')
for f in ['leaf_disease.keras', 'leaf_disease.tflite', 'labels.json', 'model_info.json']:
    mirror(MODEL_DIR / f, 'models')
    print(f'{f}: {(MODEL_DIR / f).stat().st_size / 1e6:.1f} MB')

### 5.1 Model card

Written to `research/MODEL_CARD.md` from the real results above (and copied to Drive).

In [ ]:
def pct(x):
    return f'{x * 100:.1f} %'


def md_table(df):
    head = '| ' + ' | '.join(df.columns) + ' |\n|' + '|'.join(['---'] * len(df.columns)) + '|\n'
    return head + '\n'.join('| ' + ' | '.join(str(v) for v in row) + ' |' for row in df.values.tolist())


dup_md = md_table(dup_table.reset_index().rename(columns={'index': 'Check'}))
per_class_md = md_table(per_class.round(3).reset_index().rename(columns={'index': 'Class'}))
split_md = md_table(split_table.reset_index().rename(columns={'index': 'Class'}))
conf_md = md_table(conf_df)
cmp_md = md_table(cmp_df)
llm_md = md_table(llm_df) if llm_df is not None else '_Not run (no API key)._'
field_md = f'{field_res["images"]} images' if field_res else 'not available yet'
results_md = chr(10).join('| ' + ' | '.join(str(r[k]) for k in ['Test set', 'Images', 'Accuracy', 'Macro F1']) + ' |' for r in results_df.to_dict('records'))

if OVERRIDE:
    decision_md = (f'**Final model: {FINAL}**, chosen by hand and not by the rule.\n\n'
                   f'* **Original rule:** keep v2 only if it is better on Edincik (macro F1) and its Denizli test accuracy stays at or above {DENIZLI_GATE:.0%}.\n'
                   f'* **What the rule said:** {RULE_REASON} v2 reached {pct(m2v["Denizli test: accuracy"])} on Denizli (gate {DENIZLI_GATE:.0%}).\n'
                   + (f'* **Why we overrode it:** v2 generalises much better to a new region (Edincik macro F1 {m2v["Edincik: macro F1"]:.3f} against {m1["Edincik: macro F1"]:.3f} for v1), '
                      f'which matches real use on farmers phones. The price is {abs(m2v["Denizli test: accuracy"] - m1["Denizli test: accuracy"]) * 100:.1f} points of accuracy on Denizli.\n'
                      if FINAL == 'v2' and EDI_GAIN else f'* **Why we overrode it:** chosen by hand. Note: in this run {FINAL} was not better on Edincik.' + chr(10)) +
                   f'* **Caveat:** Edincik was used for this choice, so it is not a fully untouched test. Field photos are the final check.')
else:
    decision_md = f'**Final model: {FINAL}.** {SELECTION_REASON} Rule: v2 is kept only if it is better on Edincik (macro F1) and Denizli test accuracy stays at or above {DENIZLI_GATE:.0%}.'

card = f"""# Model card: AgroPulse olive leaf disease classifier

*Generated {time.strftime('%Y-%m-%d')} by `research/02_leaf_disease_model.ipynb`. Research prototype, not a medical or agronomic device.*

## Intended use
Give a farmer a **first, quick hint** from one photo of one olive leaf: healthy, peacock spot, or Aculus olearius damage.
It is a decision aid for checking the tree more closely. **It does not replace a licensed agronomist.**

## Model
* **Final model: {FINAL}.** Backbone: **{BACKBONE}** (ImageNet weights), input 224×224. Preprocessing of the final model: {PREPROCESS_TEXT}.
* Head: global average pooling, dropout 0.3, softmax over 3 classes.
* Training (both versions): two phases (frozen backbone, then fine-tuning of the top {N_FINE_TUNE} layers), early stopping on validation loss,
  class weights, same split, same seed ({SEED}), same learning rates.
* Classes: {', '.join(CLASSES)}.
* **Which numbers are which:** all accuracy and F1 numbers in this card come from the **Keras** model, except the table "Keras vs TFLite" below. The shipped file `leaf_disease.tflite` is quantised (dynamic range) and scores slightly differently.
* Files: `models/leaf_disease.keras`, `models/leaf_disease.tflite` ({(MODEL_DIR / 'leaf_disease.tflite').stat().st_size / 1e6:.1f} MB), `models/labels.json`, `models/model_info.json`.

## Keras vs TFLite (final model {FINAL}, full Denizli test set, {n_test} images)
| Model | Accuracy on the Denizli test set |
|---|---|
| Keras (`leaf_disease.keras`) | {pct(keras_acc_full)} |
| TFLite (`leaf_disease.tflite`, shipped to the demo and the app) | {pct(lite_acc_full)} |

Keras and TFLite give the same prediction on **{pct(agree_full)}** of the {n_test} test images.

## Versions
| | v1 (first run) | v2 (experiment) |
|---|---|---|
| Preprocessing | resized (squashed) to 256×256, random crop 224 in training, centre crop when predicting | **letterbox**: padded to a square with the photo's own border colour, resized to 224×224, no crop |
| Augmentation | flips, rotation 10 %, brightness ±20 %, contrast ±20 % | flips, rotation 10 %, zoom 10 %, brightness ±30 %, contrast ±30 %, saturation ±30 %, hue ±4 %, slight blur (30 % of images), random grey background (50 % of images) |
| Training data | Denizli train (85 %) | Denizli train (85 %), same images |

No Edincik image was used for training either version. Edincik was used afterwards to choose between them, so it is not a fully untouched test.

**Comparison** (same test images):

{cmp_md}

{decision_md}

## Data
| | Denizli (Türkiye) | Edincik (Bandırma, Balıkesir) |
|---|---|---|
| Classes | Healthy, Olive Peacock Spot, Aculus Olearius | Healthy, Peacock spot |
| Images | {len(train_all) + len(test)} (official train/test split) | {len(edi)} (white background, light box) |
| Use here | training, validation, test | external test only, not used for training |
| Source / licence / citation | Kaggle "Olive Leaf Image Dataset" · [paste URL] · licence CC0 1.0 · downloaded 29 Oct 2026 | Kaggle "Olive Leaf Disease" · [paste URL] · licence unknown · downloaded 29 Oct 2026 · Diker, A., Elen, A., Közkurt, C. et al. Eur Food Res Technol (2023). https://doi.org/10.1007/s00217-023-04386-8 |

Split of the Denizli data (the official test set was never used for training or model selection; validation = 15 % of train, stratified):

{split_md}

### Duplicate check (train vs test)
{dup_md}

## Results of v1 (first run)
The tables below (results, per class, confidence, Grad-CAM, examples) describe **v1**. The v1 vs v2 table above has the numbers of both.

| Test set | Images | Accuracy | Macro F1 |
|---|---|---|---|
{results_md}

* Without the {int((~clean).sum())} Denizli test images that have a near duplicate in train: accuracy {pct(den_clean['accuracy'])}, macro F1 {den_clean['macro_f1']:.3f}.
* Greek field photos: {field_md}.

Per class, Denizli test:

{per_class_md}

Confidence (threshold {THRESHOLD:.0%}):

{conf_md}

Baseline, a general vision LLM on 60 test images:

{llm_md}

Figures (in `figures/leaf/`): `confusion_denizli_test.png`, `confusion_edincik.png`, `confusion_edincik_v2.png`, `v1_vs_v2.png`, `v2_preprocessing_edincik_photos.png`, `gradcam.png`, `examples.png`, `results_table.png`, `training_history.png`.

## Limitations
* **Turkish data only** for training (Denizli). Other regions, varieties and cultivation methods may look different.
* **Season:** the photos come from spring and summer. Autumn and winter leaves, and other stages of the disease, are not covered.
* **Leaves only.** It does not look at fruit, branches or whole trees, and it does not find olive fly, anthracnose or olive knot.
* **Edincik images are taken in a light box on a white background.** They are cleaner than field photos and are only an external check.
* **Three classes only.** A leaf with another problem (nutrient lack, sun burn, other diseases) is forced into one of the three classes.
* Photos should show **one leaf, in focus, with good light**. Blurry or dark photos give unreliable answers.
* Scores on external data are lower than on the Denizli test set; the Greek field-photo test shows the realistic level for Greece once those photos exist.
* **Not a replacement for an agronomist.**
"""
(REPO / 'MODEL_CARD.md').write_text(card, encoding='utf-8')
mirror(REPO / 'MODEL_CARD.md', '')
results_df.to_csv(REPO / 'results_table.csv', index=False)
mirror(REPO / 'results_table.csv', '')
print('Wrote', REPO / 'MODEL_CARD.md')

## 6. What to send back

Everything is on Google Drive in `MyDrive/AgroPulse/outputs/`. Please send back:

1. `figures/leaf/` (all PNG files)
2. `models/leaf_disease.tflite`, `models/labels.json` and `models/model_info.json` (and `leaf_disease.keras` if you want to keep it)
3. `MODEL_CARD.md` and `results_table.csv`
4. This notebook with its outputs: *File → Download → Download .ipynb*

In [ ]:
print('Files in the outputs folder:')
root = OUT_DRIVE if OUT_DRIVE is not None else REPO
for p in sorted(root.rglob('*')):
    if p.is_file() and p.suffix in {'.png', '.tflite', '.keras', '.json', '.md', '.csv'} and 'data' not in p.parts:
        print(f'  {p.relative_to(root)}  ({p.stat().st_size / 1e6:.2f} MB)')